# Medformer on PhysioNet 2012 ICU 48-hour Time-Series

This notebook builds a **4-hour time-bin Transformer input** from the ICU long-format time-series files and trains **Medformer** for binary in-hospital mortality prediction.

**Assumed project layout**

```text
Medformer/
├── data/
│   ├── time_series_train_a_plus_b_train_c_test*.csv
│   ├── time_series_test_a_plus_b_train_c_test*.csv
│   ├── time_series_y_train_a_plus_b_train_c_test*.csv
│   ├── time_series_y_test_a_plus_b_train_c_test*.csv
│   └── time_series_split_summary_a_plus_b_train_c_test*.csv
├── models/
│   └── Medformer.py
├── layers/
│   └── ...
└── Medformer_PhysioNetICU_48h.ipynb
```

**Main design**

- Train/development cohort: Set A+B
- External test cohort: Set C
- Temporal input: first 48 hours
- Binning: 12 bins, each 4 hours
- Features: dynamic clinical variables only
- Missingness: binary observation masks concatenated to value features
- Final input shape: `[patients, 12, 72]` = 36 value features + 36 mask features

In [ ]:
# ============================================================
# 0. Setup
# ============================================================

from pathlib import Path
import os
import sys
import json
import time
import random
import warnings

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    brier_score_loss,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
)
from sklearn.calibration import calibration_curve

import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

set_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

In [ ]:
# ============================================================
# 1. Paths and configuration
# ============================================================

PROJECT_ROOT = Path.cwd()

# If you run this notebook somewhere else, edit PROJECT_ROOT manually.
# Example:
# PROJECT_ROOT = Path("/path/to/Medformer")

# Data folder assumption from your message:
DATA_DIR = PROJECT_ROOT / "data"

# Fallback only for ChatGPT / sandbox testing. In your local run, use ./data.
if not DATA_DIR.exists() and Path("/mnt/data").exists():
    DATA_DIR = Path("/mnt/data")

OUT_DIR = PROJECT_ROOT / "transformer_temporal_outputs"
TENSOR_DIR = OUT_DIR / "data"
RESULTS_DIR = OUT_DIR / "results"
FIGURES_DIR = OUT_DIR / "figures"

for d in [OUT_DIR, TENSOR_DIR, RESULTS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data dir:", DATA_DIR)
print("Output dir:", OUT_DIR)

# Main temporal design
N_BINS = 12
BIN_MINUTES = 240  # 4 hours
VAL_SIZE = 0.20

# Cache switch: set True if you want to rebuild tensors from raw CSV every time.
FORCE_REBUILD_TENSORS = False

In [ ]:
# ============================================================
# 2. Resolve input files
# ============================================================

def find_one(data_dir: Path, pattern: str, required=True):
    matches = sorted(data_dir.glob(pattern), key=lambda p: (len(p.name), p.name))
    if not matches:
        if required:
            raise FileNotFoundError(f"No file found for pattern: {pattern} in {data_dir}")
        return None
    if len(matches) > 1:
        print(f"[Info] Multiple matches for {pattern}. Using: {matches[0].name}")
        for m in matches:
            print("   -", m.name)
    return matches[0]

train_ts_path = find_one(DATA_DIR, "time_series_train_a_plus_b_train_c_test*.csv")
test_ts_path  = find_one(DATA_DIR, "time_series_test_a_plus_b_train_c_test*.csv")
y_train_path  = find_one(DATA_DIR, "time_series_y_train_a_plus_b_train_c_test*.csv")
y_test_path   = find_one(DATA_DIR, "time_series_y_test_a_plus_b_train_c_test*.csv")
summary_path  = find_one(DATA_DIR, "time_series_split_summary_a_plus_b_train_c_test*.csv", required=False)

print("Train time-series:", train_ts_path.name)
print("Test time-series :", test_ts_path.name)
print("Train labels     :", y_train_path.name)
print("Test labels      :", y_test_path.name)
print("Summary          :", summary_path.name if summary_path else None)

In [ ]:
# ============================================================
# 3. Load labels and quick summary
# ============================================================

y_train_df = pd.read_csv(y_train_path)
y_test_df = pd.read_csv(y_test_path)

required_label_cols = {"RecordID", "in_hospital_death"}
assert required_label_cols.issubset(y_train_df.columns), y_train_df.columns
assert required_label_cols.issubset(y_test_df.columns), y_test_df.columns

record_ids_train_all = y_train_df["RecordID"].astype(int).to_numpy()
y_train_all = y_train_df["in_hospital_death"].astype(int).to_numpy()

record_ids_test = y_test_df["RecordID"].astype(int).to_numpy()
y_test = y_test_df["in_hospital_death"].astype(int).to_numpy()

print("Train/development patients:", len(record_ids_train_all))
print("External test patients    :", len(record_ids_test))
print("Train mortality prevalence:", y_train_all.mean())
print("Test mortality prevalence :", y_test.mean())

if summary_path is not None:
    display(pd.read_csv(summary_path))

In [ ]:
# ============================================================
# 4. Infer parameters and define dynamic variables
# ============================================================

STATIC_VARIABLES = {"Age", "Gender", "Height", "ICUType", "Weight"}

def infer_parameters(csv_paths, chunksize=500_000):
    params = set()
    for path in csv_paths:
        for chunk in pd.read_csv(path, usecols=["Parameter"], chunksize=chunksize):
            params.update(chunk["Parameter"].dropna().astype(str).unique())
    return sorted(params)

param_cache = TENSOR_DIR / "all_parameters.json"

if param_cache.exists() and not FORCE_REBUILD_TENSORS:
    all_parameters = json.loads(param_cache.read_text())
else:
    all_parameters = infer_parameters([train_ts_path, test_ts_path])
    param_cache.write_text(json.dumps(all_parameters, indent=2))

dynamic_vars = [p for p in all_parameters if p not in STATIC_VARIABLES]

print("All parameters:", len(all_parameters))
print("Dynamic variables used for Transformer:", len(dynamic_vars))
print(dynamic_vars)

pd.DataFrame({
    "feature_index": np.arange(len(dynamic_vars)),
    "dynamic_variable": dynamic_vars
}).to_csv(TENSOR_DIR / "dynamic_variables.csv", index=False)

In [ ]:
# ============================================================
# 5. Convert long-format time-series CSV to value + mask tensors
# ============================================================

def build_timebin_tensors(
    csv_path,
    record_ids,
    dynamic_vars,
    n_bins=12,
    bin_minutes=240,
    chunksize=500_000,
):
    """
    Convert long-format ICU records into:
      X_value: [N, n_bins, n_dynamic], float32 with NaN for missing
      X_mask : [N, n_bins, n_dynamic], float32 with 1 if observed else 0

    The function first aggregates repeated measurements within each
    RecordID × time-bin × Parameter using the mean.
    """
    record_ids = np.asarray(record_ids).astype(int)
    rid_to_idx = {int(rid): i for i, rid in enumerate(record_ids)}
    var_to_idx = {v: j for j, v in enumerate(dynamic_vars)}

    usecols = ["RecordID", "time_minutes", "Parameter", "Value_numeric"]
    agg_parts = []
    total_rows = 0
    kept_rows = 0

    print(f"Reading: {csv_path.name}")
    start = time.time()

    for chunk_idx, chunk in enumerate(pd.read_csv(csv_path, usecols=usecols, chunksize=chunksize)):
        total_rows += len(chunk)

        # Keep only selected patients and dynamic variables.
        chunk = chunk[
            chunk["RecordID"].isin(rid_to_idx)
            & chunk["Parameter"].isin(var_to_idx)
        ].copy()

        if len(chunk) == 0:
            continue

        kept_rows += len(chunk)

        chunk["bin_id"] = (chunk["time_minutes"] // bin_minutes).clip(0, n_bins - 1).astype(np.int16)
        chunk["Value_numeric"] = pd.to_numeric(chunk["Value_numeric"], errors="coerce")

        g = (
            chunk
            .dropna(subset=["Value_numeric"])
            .groupby(["RecordID", "bin_id", "Parameter"], sort=False)["Value_numeric"]
            .agg(["sum", "count"])
            .reset_index()
        )

        if len(g) > 0:
            agg_parts.append(g)

        if (chunk_idx + 1) % 5 == 0:
            print(f"  processed chunks: {chunk_idx + 1}, total rows so far: {total_rows:,}")

    if not agg_parts:
        raise ValueError(f"No valid rows found in {csv_path}")

    agg = pd.concat(agg_parts, ignore_index=True)
    agg = (
        agg
        .groupby(["RecordID", "bin_id", "Parameter"], sort=False)[["sum", "count"]]
        .sum()
        .reset_index()
    )
    agg["mean"] = agg["sum"] / agg["count"]

    X_value = np.full((len(record_ids), n_bins, len(dynamic_vars)), np.nan, dtype=np.float32)
    X_mask = np.zeros((len(record_ids), n_bins, len(dynamic_vars)), dtype=np.float32)

    i = agg["RecordID"].map(rid_to_idx).to_numpy()
    t = agg["bin_id"].astype(int).to_numpy()
    f = agg["Parameter"].map(var_to_idx).to_numpy()

    X_value[i, t, f] = agg["mean"].astype(np.float32).to_numpy()
    X_mask[i, t, f] = 1.0

    elapsed = time.time() - start
    print(f"Done: {csv_path.name}")
    print(f"  total rows: {total_rows:,}")
    print(f"  kept rows : {kept_rows:,}")
    print(f"  tensor    : {X_value.shape}")
    print(f"  elapsed   : {elapsed/60:.2f} min")

    return X_value, X_mask

# Cache paths
train_value_cache = TENSOR_DIR / "X_train_all_value.npy"
train_mask_cache  = TENSOR_DIR / "X_train_all_mask.npy"
test_value_cache  = TENSOR_DIR / "X_test_value.npy"
test_mask_cache   = TENSOR_DIR / "X_test_mask.npy"

if all(p.exists() for p in [train_value_cache, train_mask_cache, test_value_cache, test_mask_cache]) and not FORCE_REBUILD_TENSORS:
    print("Loading cached value/mask tensors...")
    X_train_all_value = np.load(train_value_cache)
    X_train_all_mask = np.load(train_mask_cache)
    X_test_value = np.load(test_value_cache)
    X_test_mask = np.load(test_mask_cache)
else:
    X_train_all_value, X_train_all_mask = build_timebin_tensors(
        train_ts_path, record_ids_train_all, dynamic_vars, n_bins=N_BINS, bin_minutes=BIN_MINUTES
    )
    X_test_value, X_test_mask = build_timebin_tensors(
        test_ts_path, record_ids_test, dynamic_vars, n_bins=N_BINS, bin_minutes=BIN_MINUTES
    )

    np.save(train_value_cache, X_train_all_value)
    np.save(train_mask_cache, X_train_all_mask)
    np.save(test_value_cache, X_test_value)
    np.save(test_mask_cache, X_test_mask)

print("Train all value:", X_train_all_value.shape)
print("Train all mask :", X_train_all_mask.shape)
print("Test value     :", X_test_value.shape)
print("Test mask      :", X_test_mask.shape)

In [ ]:
# ============================================================
# 6. Train/validation split within A+B only
# ============================================================

idx_all = np.arange(len(y_train_all))

train_idx, val_idx = train_test_split(
    idx_all,
    test_size=VAL_SIZE,
    random_state=SEED,
    stratify=y_train_all,
)

X_train_value_raw = X_train_all_value[train_idx]
X_train_mask = X_train_all_mask[train_idx]
y_train = y_train_all[train_idx]
record_ids_train = record_ids_train_all[train_idx]

X_val_value_raw = X_train_all_value[val_idx]
X_val_mask = X_train_all_mask[val_idx]
y_val = y_train_all[val_idx]
record_ids_val = record_ids_train_all[val_idx]

print("Train:", X_train_value_raw.shape, y_train.shape, y_train.mean())
print("Val  :", X_val_value_raw.shape, y_val.shape, y_val.mean())
print("Test :", X_test_value.shape, y_test.shape, y_test.mean())

pd.DataFrame({"RecordID": record_ids_train, "split": "train"}).to_csv(TENSOR_DIR / "record_ids_train.csv", index=False)
pd.DataFrame({"RecordID": record_ids_val, "split": "val"}).to_csv(TENSOR_DIR / "record_ids_val.csv", index=False)
pd.DataFrame({"RecordID": record_ids_test, "split": "test"}).to_csv(TENSOR_DIR / "record_ids_test.csv", index=False)

In [ ]:
# ============================================================
# 7. Leakage-free standardization and final Transformer tensors
# ============================================================

# Fit mean/std using TRAIN split only. Ignore missing values.
feature_mean = np.nanmean(X_train_value_raw, axis=(0, 1))
feature_std = np.nanstd(X_train_value_raw, axis=(0, 1))

# Safe fallback for features that are all missing or have zero variance in train.
feature_mean = np.where(np.isnan(feature_mean), 0.0, feature_mean)
feature_std = np.where((np.isnan(feature_std)) | (feature_std < 1e-8), 1.0, feature_std)

def scale_and_concat_mask(X_value_raw, X_mask, mean, std):
    X_scaled = (X_value_raw - mean.reshape(1, 1, -1)) / std.reshape(1, 1, -1)
    # Missing values become 0 after scaling. The mask tells the model what was missing.
    X_scaled = np.nan_to_num(X_scaled, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
    X_mask = X_mask.astype(np.float32)
    return np.concatenate([X_scaled, X_mask], axis=2).astype(np.float32)

X_train_seq = scale_and_concat_mask(X_train_value_raw, X_train_mask, feature_mean, feature_std)
X_val_seq = scale_and_concat_mask(X_val_value_raw, X_val_mask, feature_mean, feature_std)
X_test_seq = scale_and_concat_mask(X_test_value, X_test_mask, feature_mean, feature_std)

print("Final train sequence:", X_train_seq.shape)
print("Final val sequence  :", X_val_seq.shape)
print("Final test sequence :", X_test_seq.shape)

feature_names = [f"value__{v}" for v in dynamic_vars] + [f"mask__{v}" for v in dynamic_vars]

pd.DataFrame({"input_index": np.arange(len(feature_names)), "feature_name": feature_names}).to_csv(
    TENSOR_DIR / "medformer_input_feature_names.csv", index=False
)
pd.DataFrame({"dynamic_variable": dynamic_vars, "mean_train": feature_mean, "std_train": feature_std}).to_csv(
    TENSOR_DIR / "train_scaler_parameters.csv", index=False
)

np.save(TENSOR_DIR / "X_train_seq.npy", X_train_seq)
np.save(TENSOR_DIR / "X_val_seq.npy", X_val_seq)
np.save(TENSOR_DIR / "X_test_seq.npy", X_test_seq)
np.save(TENSOR_DIR / "y_train.npy", y_train)
np.save(TENSOR_DIR / "y_val.npy", y_val)
np.save(TENSOR_DIR / "y_test.npy", y_test)

In [ ]:
# ============================================================
# 8. PyTorch Dataset and DataLoader
# ============================================================

class ICUSequenceDataset(Dataset):
    def __init__(self, X, y, record_ids=None):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        self.record_ids = None if record_ids is None else np.asarray(record_ids)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

BATCH_SIZE = 64

train_loader = DataLoader(
    ICUSequenceDataset(X_train_seq, y_train, record_ids_train),
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=False,
)

val_loader = DataLoader(
    ICUSequenceDataset(X_val_seq, y_val, record_ids_val),
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False,
)

test_loader = DataLoader(
    ICUSequenceDataset(X_test_seq, y_test, record_ids_test),
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False,
)

print("Batches:", len(train_loader), len(val_loader), len(test_loader))

In [ ]:
# ============================================================
# 9. Import Medformer model
# ============================================================

# This notebook should be run from the root of the cloned Medformer repo.
# If not, change PROJECT_ROOT above or append the repo path manually.

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

try:
    from models.Medformer import Model as MedformerModel
except Exception as e:
    raise ImportError(
        "Could not import models.Medformer. Run this notebook from the Medformer repo root, "
        "or set PROJECT_ROOT to the cloned Medformer directory."
    ) from e

from types import SimpleNamespace

config = SimpleNamespace(
    task_name="classification",
    pred_len=0,
    output_attention=False,

    # Data-dependent values
    enc_in=X_train_seq.shape[2],
    seq_len=X_train_seq.shape[1],
    num_class=2,

    # Medformer model size
    d_model=64,
    n_heads=4,
    e_layers=2,
    d_ff=128,
    dropout=0.1,
    activation="gelu",

    # Medformer-specific settings
    patch_len_list="2,4,6",
    augmentations="none",
    single_channel=False,
    no_inter_attn=False,
)

model = MedformerModel(config).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model.__class__.__name__)
print("Trainable parameters:", f"{n_params:,}")
print("Input shape expected: [batch, seq_len, enc_in] =", [BATCH_SIZE, config.seq_len, config.enc_in])

In [ ]:
# ============================================================
# 10. Training and evaluation utilities
# ============================================================

def predict_proba(model, loader):
    model.eval()
    all_logits = []
    all_y = []

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(DEVICE)
            logits = model(X_batch, None, None, None)
            all_logits.append(logits.detach().cpu())
            all_y.append(y_batch.detach().cpu())

    logits = torch.cat(all_logits, dim=0)
    y_true = torch.cat(all_y, dim=0).numpy()
    probs = torch.softmax(logits, dim=1).numpy()
    p_mortality = probs[:, 1]
    return y_true, p_mortality, probs, logits.numpy()

def compute_metrics(y_true, p_mortality, threshold=0.5):
    y_pred = (p_mortality >= threshold).astype(int)

    out = {
        "ROC_AUC": roc_auc_score(y_true, p_mortality),
        "PR_AUC": average_precision_score(y_true, p_mortality),
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "Brier": brier_score_loss(y_true, p_mortality),
    }
    return out

def evaluate_loader(model, loader, threshold=0.5):
    y_true, p_mortality, probs, logits = predict_proba(model, loader)
    metrics = compute_metrics(y_true, p_mortality, threshold=threshold)
    return metrics, y_true, p_mortality, probs, logits

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    losses = []

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)

        optimizer.zero_grad()
        logits = model(X_batch, None, None, None)
        loss = criterion(logits, y_batch)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=4.0)
        optimizer.step()

        losses.append(loss.item())

    return float(np.mean(losses))

In [ ]:
# ============================================================
# 11. Train Medformer
# ============================================================

LEARNING_RATE = 1e-4
MAX_EPOCHS = 50
PATIENCE = 8

# Class imbalance handling.
n_pos = int((y_train == 1).sum())
n_neg = int((y_train == 0).sum())
class_weights = torch.tensor([1.0, n_neg / max(n_pos, 1)], dtype=torch.float32).to(DEVICE)
print("Class counts:", {"negative": n_neg, "positive": n_pos})
print("Class weights:", class_weights.detach().cpu().numpy())

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

best_val_auprc = -np.inf
best_state = None
epochs_without_improvement = 0
history = []

start_train = time.time()

for epoch in range(1, MAX_EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
    val_metrics, _, _, _, _ = evaluate_loader(model, val_loader)

    row = {"epoch": epoch, "train_loss": train_loss, **{f"val_{k}": v for k, v in val_metrics.items()}}
    history.append(row)

    print(
        f"Epoch {epoch:03d} | "
        f"loss={train_loss:.4f} | "
        f"val_AUPRC={val_metrics['PR_AUC']:.4f} | "
        f"val_AUROC={val_metrics['ROC_AUC']:.4f} | "
        f"val_F1={val_metrics['F1']:.4f} | "
        f"val_Brier={val_metrics['Brier']:.4f}"
    )

    # Early stopping by validation PR-AUC because mortality is imbalanced.
    if val_metrics["PR_AUC"] > best_val_auprc + 1e-5:
        best_val_auprc = val_metrics["PR_AUC"]
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= PATIENCE:
        print(f"Early stopping at epoch {epoch}.")
        break

elapsed = time.time() - start_train
print(f"Training finished in {elapsed/60:.2f} min.")

history_df = pd.DataFrame(history)
history_df.to_csv(RESULTS_DIR / "medformer_training_history.csv", index=False)

# Restore best validation model.
if best_state is not None:
    model.load_state_dict(best_state)
    torch.save(best_state, RESULTS_DIR / "medformer_best_state.pt")

display(history_df.tail())

In [ ]:
# ============================================================
# 12. Final external test evaluation on Set C
# ============================================================

test_metrics, y_true_test, p_test, probs_test, logits_test = evaluate_loader(model, test_loader)

metrics_df = pd.DataFrame([{"split": "external_test_C", **test_metrics}])
metrics_df.to_csv(RESULTS_DIR / "medformer_external_test_metrics.csv", index=False)

pred_df = pd.DataFrame({
    "RecordID": record_ids_test,
    "y_true": y_true_test,
    "p_mortality": p_test,
    "y_pred_0p5": (p_test >= 0.5).astype(int),
    "split": "test_C",
})
pred_df.to_csv(RESULTS_DIR / "medformer_external_test_predictions.csv", index=False)

print("External test metrics:")
display(metrics_df)

print("Confusion matrix at threshold 0.5:")
print(confusion_matrix(y_true_test, (p_test >= 0.5).astype(int)))

In [ ]:
# ============================================================
# 13. Plot ROC, PR, and calibration curves
# ============================================================

# ROC curve
fpr, tpr, _ = roc_curve(y_true_test, p_test)
plt.figure(figsize=(5.5, 4.5))
plt.plot(fpr, tpr, label=f"Medformer (AUC = {test_metrics['ROC_AUC']:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", label="Chance")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("External Test ROC Curve")
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "medformer_external_test_roc_curve.png", dpi=300)
plt.show()

# PR curve
precision, recall, _ = precision_recall_curve(y_true_test, p_test)
baseline_prev = y_true_test.mean()
plt.figure(figsize=(5.5, 4.5))
plt.plot(recall, precision, label=f"Medformer (AUPRC = {test_metrics['PR_AUC']:.3f})")
plt.axhline(baseline_prev, linestyle="--", label=f"Prevalence = {baseline_prev:.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("External Test Precision-Recall Curve")
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "medformer_external_test_pr_curve.png", dpi=300)
plt.show()

# Calibration curve
prob_true, prob_pred = calibration_curve(y_true_test, p_test, n_bins=10, strategy="quantile")
plt.figure(figsize=(5.5, 4.5))
plt.plot(prob_pred, prob_true, marker="o", label=f"Medformer (Brier = {test_metrics['Brier']:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", label="Perfect calibration")
plt.xlabel("Mean Predicted Mortality Risk")
plt.ylabel("Observed Mortality Rate")
plt.title("External Test Calibration Curve")
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "medformer_external_test_calibration_curve.png", dpi=300)
plt.show()

In [ ]:
# ============================================================
# 14. Time-bin permutation importance
# ============================================================

def evaluate_array(model, X_array, y_array, batch_size=128):
    loader = DataLoader(
        ICUSequenceDataset(X_array, y_array),
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
    )
    metrics, y_true, p, _, _ = evaluate_loader(model, loader)
    return metrics

def timebin_permutation_importance(
    model,
    X_base,
    y_base,
    baseline_metrics,
    n_repeats=3,
    seed=SEED,
):
    rows = []
    rng = np.random.default_rng(seed)

    for bin_id in range(X_base.shape[1]):
        for rep in range(n_repeats):
            X_mod = X_base.copy()
            perm = rng.permutation(X_mod.shape[0])
            X_mod[:, bin_id, :] = X_mod[perm, bin_id, :]

            m = evaluate_array(model, X_mod, y_base)
            rows.append({
                "bin_id": bin_id,
                "time_window": f"{bin_id * 4}-{(bin_id + 1) * 4}h",
                "repeat": rep,
                "ROC_AUC_permuted": m["ROC_AUC"],
                "PR_AUC_permuted": m["PR_AUC"],
                "Brier_permuted": m["Brier"],
                "ROC_AUC_drop": baseline_metrics["ROC_AUC"] - m["ROC_AUC"],
                "PR_AUC_drop": baseline_metrics["PR_AUC"] - m["PR_AUC"],
                "Brier_increase": m["Brier"] - baseline_metrics["Brier"],
            })

    return pd.DataFrame(rows)

timebin_imp_df = timebin_permutation_importance(
    model,
    X_test_seq,
    y_test,
    baseline_metrics=test_metrics,
    n_repeats=3,
)

timebin_summary_df = (
    timebin_imp_df
    .groupby(["bin_id", "time_window"], as_index=False)
    .agg(
        mean_PR_AUC_drop=("PR_AUC_drop", "mean"),
        std_PR_AUC_drop=("PR_AUC_drop", "std"),
        mean_ROC_AUC_drop=("ROC_AUC_drop", "mean"),
        std_ROC_AUC_drop=("ROC_AUC_drop", "std"),
        mean_Brier_increase=("Brier_increase", "mean"),
        std_Brier_increase=("Brier_increase", "std"),
    )
)

timebin_imp_df.to_csv(RESULTS_DIR / "timebin_permutation_importance_repeats.csv", index=False)
timebin_summary_df.to_csv(RESULTS_DIR / "timebin_permutation_importance_summary.csv", index=False)

display(timebin_summary_df)

In [ ]:
# ============================================================
# 15. Plot time-bin permutation importance
# ============================================================

plt.figure(figsize=(7.5, 4.5))
plt.bar(
    timebin_summary_df["time_window"],
    timebin_summary_df["mean_PR_AUC_drop"],
    yerr=timebin_summary_df["std_PR_AUC_drop"].fillna(0),
    capsize=3,
)
plt.xlabel("ICU Time Window")
plt.ylabel("Mean PR-AUC Drop After Permutation")
plt.title("Time-bin Permutation Importance")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "timebin_permutation_importance_pr_auc_drop.png", dpi=300)
plt.show()

In [ ]:
# ============================================================
# 16. Feature × time-bin importance heatmap
# ============================================================

# This step evaluates 12 × 36 = 432 perturbations, so it may take longer.
RUN_FEATURE_TIME_IMPORTANCE = True
FEATURE_TIME_REPEATS = 1

def feature_time_permutation_importance(
    model,
    X_base,
    y_base,
    dynamic_vars,
    baseline_metrics,
    n_repeats=1,
    seed=SEED,
):
    n_dynamic = len(dynamic_vars)
    rows = []
    rng = np.random.default_rng(seed)

    for bin_id in range(X_base.shape[1]):
        print(f"Time bin {bin_id + 1}/{X_base.shape[1]}")
        for feat_idx, feat_name in enumerate(dynamic_vars):
            for rep in range(n_repeats):
                X_mod = X_base.copy()
                perm = rng.permutation(X_mod.shape[0])

                value_col = feat_idx
                mask_col = feat_idx + n_dynamic

                # Permute both value and observation mask for this variable at this time bin.
                X_mod[:, bin_id, value_col] = X_mod[perm, bin_id, value_col]
                X_mod[:, bin_id, mask_col] = X_mod[perm, bin_id, mask_col]

                m = evaluate_array(model, X_mod, y_base)
                rows.append({
                    "bin_id": bin_id,
                    "time_window": f"{bin_id * 4}-{(bin_id + 1) * 4}h",
                    "feature": feat_name,
                    "repeat": rep,
                    "ROC_AUC_drop": baseline_metrics["ROC_AUC"] - m["ROC_AUC"],
                    "PR_AUC_drop": baseline_metrics["PR_AUC"] - m["PR_AUC"],
                    "Brier_increase": m["Brier"] - baseline_metrics["Brier"],
                })

    return pd.DataFrame(rows)

if RUN_FEATURE_TIME_IMPORTANCE:
    feature_time_imp_df = feature_time_permutation_importance(
        model,
        X_test_seq,
        y_test,
        dynamic_vars,
        baseline_metrics=test_metrics,
        n_repeats=FEATURE_TIME_REPEATS,
    )

    feature_time_imp_df.to_csv(RESULTS_DIR / "feature_time_permutation_importance.csv", index=False)

    feature_time_matrix = (
        feature_time_imp_df
        .groupby(["feature", "bin_id"])["PR_AUC_drop"]
        .mean()
        .unstack("bin_id")
        .reindex(dynamic_vars)
    )

    feature_time_matrix.to_csv(RESULTS_DIR / "feature_time_importance_matrix_pr_auc_drop.csv")
    display(feature_time_matrix.head())
else:
    print("Skipped feature × time-bin importance.")

In [ ]:
# ============================================================
# 17. Plot feature × time-bin importance heatmap
# ============================================================

if RUN_FEATURE_TIME_IMPORTANCE:
    feature_time_matrix = pd.read_csv(
        RESULTS_DIR / "feature_time_importance_matrix_pr_auc_drop.csv",
        index_col=0,
    )

    plt.figure(figsize=(9, 8))
    im = plt.imshow(feature_time_matrix.values, aspect="auto")
    plt.colorbar(im, label="Mean PR-AUC Drop")
    plt.xticks(
        ticks=np.arange(N_BINS),
        labels=[f"{i*4}-{(i+1)*4}h" for i in range(N_BINS)],
        rotation=45,
        ha="right",
    )
    plt.yticks(ticks=np.arange(len(feature_time_matrix.index)), labels=feature_time_matrix.index)
    plt.xlabel("ICU Time Window")
    plt.ylabel("Clinical Variable")
    plt.title("Feature × Time-bin Permutation Importance")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "feature_time_importance_heatmap_pr_auc_drop.png", dpi=300)
    plt.show()

In [ ]:
# ============================================================
# 18. Save compact manuscript-ready summary
# ============================================================

summary = {
    "model": "Medformer",
    "task": "In-hospital mortality prediction",
    "train_cohort": "Set A+B",
    "external_test_cohort": "Set C",
    "time_window_hours": 48,
    "bin_size_hours": 4,
    "n_time_bins": N_BINS,
    "n_dynamic_variables": len(dynamic_vars),
    "n_input_channels": X_train_seq.shape[2],
    "train_patients": len(y_train),
    "validation_patients": len(y_val),
    "test_patients": len(y_test),
    **{f"test_{k}": float(v) for k, v in test_metrics.items()},
}

summary_df = pd.DataFrame([summary])
summary_df.to_csv(RESULTS_DIR / "medformer_manuscript_summary.csv", index=False)

display(summary_df)

print("Saved outputs to:")
print("  Data   :", TENSOR_DIR)
print("  Results:", RESULTS_DIR)
print("  Figures:", FIGURES_DIR)

## Notes for manuscript writing

Suggested methods text:

> Raw ICU time-series measurements from the first 48 hours were converted into 12 four-hour time bins. Within each patient-time bin, repeated measurements of each dynamic clinical variable were aggregated using the mean. Binary observation masks were concatenated to standardized value features to preserve measurement availability. Standardization parameters were estimated using the training split of Set A+B only and then applied unchanged to validation and external Set C data. The resulting patient-level sequence was used as input to a Medformer-based temporal classifier for in-hospital mortality prediction.

Suggested results focus:

- Report external Set C ROC-AUC, PR-AUC, F1, and Brier score.
- Use calibration curve because this is a risk prediction task.
- Use time-bin permutation importance to show which 4-hour windows carry the strongest mortality signal.
- Use feature × time-bin heatmap as temporal evidence that can later be compared with the Markov-chain trajectory analysis.